---
title: Feature relationships
execute:
  enabled: true
---

`q.plot.correlation` creates an interactive Plotly scatterplot matrix for exploring relationships among numeric features. Box or lasso selection is linked across every panel, and the observation timestamp is included in the hover context.

This example uses qrt's bundled AAPL and SPY histories, so it executes without a network connection. AAPL supplies the feature and forward-return observations; SPY's trend defines the surrounding market regime.

## Build market features

The feature set combines momentum, volatility, and relative volume. The five-session forward return is evaluation context only: it is deliberately excluded from the plotted feature dimensions to avoid presenting future information as a model input.

In [4]:
import numpy as np
import pandas as pd

import qrt as q

aapl = q.data.datasets.load("aapl")
spy = q.data.datasets.load("spy")
common_sessions = aapl.index.intersection(spy.index)
aapl = aapl.loc[common_sessions]
spy = spy.loc[common_sessions]

aapl_returns = q.indicator.returns(aapl["close"])
spy_trend = q.indicator.ema(spy["close"], 20).div(
    q.indicator.sma(spy["close"], 100)
).sub(1)
regime = pd.Series(
    np.select(
        [spy_trend < -0.01, spy_trend > 0.01],
        ["risk-off", "risk-on"],
        default="neutral",
    ),
    index=spy.index,
    name="regime",
)

features = pd.DataFrame(
    {
        "momentum_5d": q.indicator.momentum(aapl["close"], window=5),
        "momentum_20d": q.indicator.momentum(aapl["close"], window=20),
        "volatility_20d": q.indicator.rolling_volatility(aapl_returns, window=20),
        "volume_ratio_20d": q.indicator.volume_ratio(aapl["volume"], window=20),
        "regime": regime,
        "forward_return_5d": q.indicator.returns(aapl["close"], periods=5).shift(-5),
        "symbol": "AAPL",
    }
).dropna().iloc[-750:]

features.tail()

,momentum_5d,momentum_20d,volatility_20d,volume_ratio_20d,regime,forward_return_5d,symbol
datetime,,,,,,,
2026-07-10,0.021676,0.081419,0.022139,0.694532,risk-on,0.058417,AAPL
2026-07-13,0.014872,0.073335,0.022029,0.880219,risk-on,0.029246,AAPL
2026-07-14,0.013520,0.081510,0.021750,0.739389,risk-on,0.040907,AAPL
2026-07-15,0.045024,0.104851,0.023014,1.193199,risk-on,-0.004916,AAPL
2026-07-16,0.053887,0.113688,0.023163,1.198381,risk-on,-0.034808,AAPL


## Color by market regime

Use `color_discrete_map` when categories have fixed meaning. Here risk-off is always red, neutral is gray, and risk-on is green, regardless of the order in which those regimes appear in the data. For a simple rotating palette instead, pass `color_discrete_sequence=["#color1", "#color2", ...]`.

The helper uses high-contrast markers, dark boxed subplot axes, and outward ticks by default. The example makes them stronger with `marker_*`, `axis_color`, `axis_line_width`, `tick_length`, and `tick_width`. Setting `diagonal=True` displays each feature against itself. Legend entries can be hidden or isolated, and selecting observations in one panel highlights them throughout the matrix.

In [5]:
feature_columns = [
    "momentum_5d",
    "momentum_20d",
    "volatility_20d",
    "volume_ratio_20d",
]

regime_colors = {
    "risk-off": "#D1495B",
    "neutral": "#8D99AE",
    "risk-on": "#2A9D8F",
}

fig = q.plot.correlation(
    features,
    columns=feature_columns,
    color="regime",
    color_discrete_map=regime_colors,
    hover_data=["symbol", "forward_return_5d"],
    diagonal=True,
    marker_size=8,
    marker_opacity=1.0,
    marker_line_width=0.8,
    marker_line_color="#1F2937",
    axis_color="#1F2937",
    axis_line_width=1.4,
    tick_length=6,
    tick_width=1.4,
    title="AAPL feature relationships by SPY market regime",
)
fig.show()

## Color by forward outcome

Use `color_continuous_scale` for a numeric color variable. It accepts Plotly scale names such as `"RdYlGn"`, `"RdBu"`, `"Viridis"`, or `"Cividis"`; append `_r` to reverse a named scale, for example `"RdBu_r"`. An explicit sequence such as `["#B2182B", "#F7F7F7", "#2166AC"]` also works.

Since these outcomes cross zero, QRT centers the selected scale at zero. This example uses red for losses and blue for gains, keeps markers fully opaque for contrast, and enables the diagonal.

In [6]:
fig = q.plot.correlation(
    features,
    columns=feature_columns,
    color="forward_return_5d",
    color_continuous_scale="RdBu",  # use "RdBu_r" to reverse it
    hover_data=["symbol", "regime"],
    triangle="full",
    diagonal=True,
    marker_opacity=1.0,
    marker_line_color="#1F2937",
    title="AAPL feature relationships colored by 5-day forward return",
)
fig.show()

The default lower triangle removes duplicate panels. Use `triangle="upper"` or `triangle="full"` when presentation or comparison needs the other panels. In research workflows, useful categorical colors include asset, volatility regime, predicted action, and train/validation/test split; useful continuous colors include forward return, realized volatility, model confidence, and sample weight.